In [2]:
from pprint import pprint
import os.path
import os
import pandas as pd
from datetime import datetime


lst_path = []
output_path = r"output.xlsx"

path_folder = r"C:\Users\Admin\Downloads\Imported data-20241016T071429Z-001\Imported data"

for root, dirs, files in os.walk(path_folder):
    # print(dirs)
    for dir_child in dirs:
        child_path = os.path.join(root, dir_child)
        for _, _, files_child in os.walk(child_path):
            for file in files_child:
                path = os.path.join(child_path, file)
                lst_path.append(path)

pprint(lst_path)

['C:\\Users\\Admin\\Downloads\\Imported data-20241016T071429Z-001\\Imported '
 'data\\Bodymist_Q32024_Thắng\\Bodymist_Q32024_Thắng.xlsx',
 'C:\\Users\\Admin\\Downloads\\Imported data-20241016T071429Z-001\\Imported '
 'data\\Bàn chải điện_Q32024_Thắng\\Bàn chải điện_Q32024_Thắng.xlsx',
 'C:\\Users\\Admin\\Downloads\\Imported data-20241016T071429Z-001\\Imported '
 'data\\Collagen_Q32024_Thắng\\Collagen_Q32024_Thắng.xlsx',
 'C:\\Users\\Admin\\Downloads\\Imported data-20241016T071429Z-001\\Imported '
 'data\\DDVS_Q32024_Thắng\\Dung_dich_ve_sinh_Q32024_Thắng_20240927_175424.xlsx',
 'C:\\Users\\Admin\\Downloads\\Imported data-20241016T071429Z-001\\Imported '
 'data\\Dầu gội, xả_Q32024_Thắng\\Dầu gội xả_Q32024_Thắng.xlsx',
 'C:\\Users\\Admin\\Downloads\\Imported data-20241016T071429Z-001\\Imported '
 'data\\Kem chống nắng_Q32024_Thắng\\Kem chống nắng_Q32024_Thắng.xlsx',
 'C:\\Users\\Admin\\Downloads\\Imported data-20241016T071429Z-001\\Imported '
 'data\\Kem dưỡng_Q32024_Thắng\\Kem dưỡng_Q320

In [3]:
def get_inf_file_audit_report(name_cate, path_file):
    import pandas as pd
    from datetime import datetime
    
    df_raw = pd.read_excel(path_file)
    df_raw['cleaned_brand'] = df_raw['cleaned_brand'].fillna('').astype(str)
    current_time = datetime.now()
    df_raw.copy()
    
    # df_concat = pd.concat([df_raw, df_raw], ignore_index=True)
    df_raw["sum_sales"] = df_raw[[
        "sale_202201",
        "sale_202202",
        "sale_202203",
        "sale_202204",
        "sale_202205",
        "sale_202206",
        "sale_202207",
        "sale_202208",
        "sale_202209",
        "sale_202210",
        "sale_202211",
        "sale_202212",
        "sale_202301",
        "sale_202302",
        "sale_202303",
        "sale_202304",
        "sale_202305",
        "sale_202306",
        "sale_202307",
        "sale_202308",
        "sale_202309",
        "sale_202310",
        "sale_202311",
        "sale_202312",
        "sale_202401",
        "sale_202402",
        "sale_202403",
        "sale_202404",
        "sale_202405",
        "sale_202406",
        "sale_202407",
        "sale_202408",
        "sale_202409",
        "sale_202410",
        "sale_202411",
        "sale_202412",
        "sale_202501",
        "sale_202502",
        "sale_202503",
        "sale_202504",
        "sale_202505",
        "sale_202506",
        "sale_202507",
        "sale_202508",
        "sale_202509",
        "sale_202510",
        "sale_202511",
        "sale_202512"
    ]].sum(axis=1)
    
    # Tính số dòng lấy
    df_raw["no_sales"] = df_raw["sum_sales"] == 0
    number_select =  len(df_raw[df_raw["is_valid_report"] == True])
    number_remove =  len(df_raw[df_raw["is_valid_report"] == False])
    
    # Tính các sản phẩm không có SL
    df_raw["no_sales"] = df_raw["sum_sales"] == 0
    
    # Tính số lượng brand trong file
    df_raw["cleaned_brand"].nunique()
    
    # Tính số lượng product_id trong file
    df_raw["no_brand"] = df_raw["cleaned_brand"] == "no brand"
    no_brand_row = df_raw[df_raw["no_brand"] == True]
    
    df_raw["no_brand"] = df_raw["cleaned_brand"] == "chưa biết"
    chua_biet_row = df_raw[df_raw["no_brand"] == True]
    
    df_raw["no_brand"] = df_raw["cleaned_brand"] == "0"
    khong_row = df_raw[df_raw["no_brand"] == True]
    
    # Thống kê brand
    df_raw['revenue_in_range'] = df_raw['revenue_in_range'].fillna(0).astype(int)
    df_static = df_raw[df_raw["is_valid_report"] == True]
    revenue_all = df_static['revenue_in_range'].sum()
    sales_all = df_static['sale_in_range'].sum()
    
    static_br = df_static.pivot_table(index="cleaned_brand", 
                    values="revenue_in_range",
                    aggfunc = "sum"
                    ).sort_values(by = "revenue_in_range", ascending= False).head(6).reset_index()
    
    static_br["per_of_total"] = ((static_br["revenue_in_range"] / revenue_all) * 100).round(0).astype(int)
    static_br["per_of_total"] = static_br["per_of_total"].astype(str) + "%"
    
    static_br["revenue_in_range"] = static_br["revenue_in_range"].apply(lambda x: f"{int(x):,}")
    
    # Có điền thiếu brand không ? có clean đủ hay chưa ?
    df_raw_valid = df_raw[df_raw["is_valid_report"] == True]
    df_null = df_raw_valid[df_raw_valid["is_valid_report"].isnull()]

    return f'''
Cập nhật ngày {current_time.strftime("%d-%m-%Y %H:%M")}
Thống kê file: {name_cate}

1. Tổng quan:
DS: {revenue_all:,} / SL: {sales_all:,}
File raw: Số dòng {int(df_raw.shape[0]):,} / Số cột: {int(df_raw.shape[1]):,}
Số dòng lấy: {number_select:,}
Số dòng loại: {number_remove:,}

2. Số dòng trùng
Số dòng trùng pro_id: {(len(df_raw) - df_raw["product_base_id"].nunique()):,}

3. Số liệu
Số dòng không có SL/ DS: {df_raw["no_sales"].sum():,}
Số dòng clean thiếu: {len(df_null)}

4. Thống kê brand
Có {df_raw["cleaned_brand"].nunique():,} brand unique
Số lượng brand "No brand": {len(no_brand_row):,}
Số lượng brand "Chưa biết": {len(chua_biet_row):,}
Số lượng brand "0": {len(khong_row):,}

Top 5 brand
{static_br}
'''

In [4]:
from tqdm import tqdm

df = pd.DataFrame()
df_ = pd.DataFrame()

for path_excel in tqdm(lst_path):
    name_ = os.path.split(path_excel)[1]
    cate = name_.split("_")[0] 
    inf_cat = None
    try:
        inf_cat = get_inf_file_audit_report(cate, path_excel)
    except Exception as e:
        print(e)
        
    temp_df = pd.DataFrame({
        "name": [cate],
        "inf_cat": [inf_cat]
    })

    df_ = pd.concat([df_, temp_df], ignore_index=True)
    
    
df_


 94%|█████████▎| 29/31 [37:55<02:00, 60.34s/it]  

can only concatenate str (not "int") to str


100%|██████████| 31/31 [38:08<00:00, 73.82s/it]

[Errno 2] No such file or directory: 'C:\\Users\\Admin\\Downloads\\Imported data-20241016T071429Z-001\\Imported data\\Xịt khoáng_Q32024_Thắng\\~$Xịt khoáng_Q32024_Thắng.xlsx'


,name,inf_cat
0,Bodymist,\nCập nhật ngày 17-10-2024 14:37\nThống kê fil...
1,Bàn chải điện,\nCập nhật ngày 17-10-2024 14:37\nThống kê fil...
2,Collagen,\nCập nhật ngày 17-10-2024 14:37\nThống kê fil...
3,Dung,\nCập nhật ngày 17-10-2024 14:38\nThống kê fil...
4,Dầu gội xả,\nCập nhật ngày 17-10-2024 14:40\nThống kê fil...
5,Kem chống nắng,\nCập nhật ngày 17-10-2024 14:42\nThống kê fil...
6,Kem dưỡng,\nCập nhật ngày 17-10-2024 14:47\nThống kê fil...
7,Kem nền,\nCập nhật ngày 17-10-2024 14:48\nThống kê fil...
8,Kem và sữa dưỡng thể,\nCập nhật ngày 17-10-2024 14:50\nThống kê fil...
9,Khử mùi cơ thể,\nCập nhật ngày 17-10-2024 14:51\nThống kê fil...


In [5]:
with pd.ExcelWriter(output_path, engine='xlsxwriter',
                    engine_kwargs={'options': {'strings_to_urls': False}}) as writer:
    df_.to_excel(writer, index=False)